In [6]:
import pandas as pd
import numpy as np
from collections import Counter
from datetime import datetime

# --- Load the dataset ---
df = pd.read_csv('dataset/lyrics_annotated_line.csv') 
# --- Extract year and decade ---
def extract_year(date_str):
    try:
        return pd.to_datetime(date_str).year
    except:
        return np.nan

df['publicationYear'] = df['publicationDate'].apply(extract_year)
df['decade'] = (df['publicationYear'] // 10 * 10).astype('Int64').astype('string') + 's'

# --- Language fallback ---
df['language_final'] = df['language_detect'].fillna(df['language'])

# --- Mean VAD scores ---
vad_cols = ['valence', 'valence_predicted', 'valence_lexicon']
df['valence_mean'] = df[vad_cols].mean(axis=1, skipna=True)

arousal_cols = ['arousal', 'arousal_predicted', 'arousal_lexicon']
df['arousal_mean'] = df[arousal_cols].mean(axis=1, skipna=True)

df['dominance_mean'] = df['dominance_lexicon']

# --- Episode mode ---
def most_common_episode(episodes):
    if isinstance(episodes, str):
        episodes = episodes.strip("[]").replace("'", "").split(", ")
    return Counter(episodes).most_common(1)[0][0] if episodes else np.nan

df['episode_mode'] = df['episode'].apply(most_common_episode)

# --- Line count ---
df['line_count'] = df['lyrics'].astype(str).apply(lambda x: len(x.split('<br>')))

# --- Final subset of columns for trend analysis ---
output_cols = [
    'publicationYear', 'decade', 'language_final', 'isClassic', 'explicitLyrics',
    'valence_mean', 'arousal_mean', 'dominance_mean', 'episode_mode',
    'line_count', 'position'
]

output_df = df[output_cols].rename(columns={'language_final': 'language'})

# --- Export to new CSV ---
output_df.to_csv('lyrical_emotion_trends.csv', index=False)
print("✅ Saved to 'lyrical_emotion_trends.csv'")

/var/folders/qf/5crdbb_x01j7jrrtsg93r4dc0000gn/T/ipykernel_94651/1961404866.py:7: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('dataset/lyrics_annotated_line.csv')  # Replace with your actual filename


✅ Saved to 'lyrical_emotion_trends.csv'


In [9]:
ls

LICENSE                                dataset/
Lyrical-Emotion-across-Cultures.ipynb  lyrical_emotion_trends.csv
README.md                              milestone 1 report.pdf
data_prep.ipynb


In [12]:
print("Column Headings:", df.columns.tolist())
print("Shape:", df.shape)

Column Headings: ['publicationYear', 'decade', 'language', 'isClassic', 'explicitLyrics', 'valence_mean', 'arousal_mean', 'dominance_mean', 'episode_mode', 'line_count', 'position']
Shape: (1679972, 11)


In [16]:
import pandas as pd
import re
from tqdm import tqdm

# Load your datasets
lyrics_df = pd.read_csv('dataset/lyrics_annotated_line.csv', usecols=['lyrics'])

# Define your lexicons
love_words = {'adore', 'affection', 'amour', 'beloved', 'cherish', 'darling', 'devotion', 'embrace', 'fondness', 'infatuation', 'kiss', 'passion', 'romance', 'sweetheart', 'tenderness'}
swear_words = {'asshole', 'bastard', 'bitch', 'bloody', 'bollocks', 'bugger', 'crap', 'damn', 'fuck', 'motherfucker', 'piss', 'shit', 'slut', 'twat', 'wanker'}
money_words = {'affluence', 'bank', 'cash', 'coin', 'currency', 'dollar', 'economy', 'finance', 'gold', 'income', 'investment', 'money', 'profit', 'rich', 'wealth'}
sex_words = {'arousal', 'climax', 'erotic', 'fornicate', 'intercourse', 'libido', 'lust', 'nude', 'orgasm', 'passion', 'sensual', 'sexual', 'titillate', 'virgin', 'voyeur'}
neutral_words = {'book', 'chair', 'door', 'house', 'lamp', 'paper', 'pen', 'shoe', 'table', 'window'}

# Function to count words in a given text
def count_words(text, word_set):
    words = re.findall(r'\b\w+\b', str(text).lower())  # Tokenize and convert to lowercase
    return sum(1 for word in words if word in word_set)

# Initialize tqdm to work with pandas
tqdm.pandas(desc="Processing lyrics")

# Apply the function to each row in the dataframe with a progress bar
df['love_word_count'] = lyrics_df['lyrics'].progress_apply(lambda x: count_words(x, love_words))
df['swear_word_count'] = lyrics_df['lyrics'].progress_apply(lambda x: count_words(x, swear_words))
df['money_word_count'] = lyrics_df['lyrics'].progress_apply(lambda x: count_words(x, money_words))
df['sex_word_count'] = lyrics_df['lyrics'].progress_apply(lambda x: count_words(x, sex_words))
df['neutral_word_count'] = lyrics_df['lyrics'].progress_apply(lambda x: count_words(x, neutral_words))

# Save the updated dataframe to a new CSV file
df.to_csv('lyrical_emotion_trends.csv', index=False)

Processing lyrics: 100%|██████████| 1679972/1679972 [01:09<00:00, 24168.84it/s]
